# 🏞️ Jawalgaon Dam — Dam Break Analysis (DBA)
## Medium Irrigation Project | Barshi, Solapur, Maharashtra
### Breach Hydrograph Analysis — Overtopping (OVTP) & Piping (PIPG) Failure Modes

---
**Plot Order:**
```
1. OVERTOPPING (OVTP) FAILURE
   1.1  HW, TW Elevation & Flow
   1.2  Breach Width
   1.3  Breach Velocity
   1.4  Downstream Hydrograph

2. PIPING (PIPG) FAILURE
   2.1  HW, TW Elevation & Flow
   2.2  Breach Width
   2.3  Breach Velocity
   2.4  Downstream Hydrograph
```
---
**Simulation:** 72 hours | OVTP @ 5-sec intervals (51,841 records) | PIPG @ 1-sec intervals (259,201 records)  
**Reference:** DBA-JWLG-2026 | SPF (Standard Project Flood)

In [1]:
# ════════════════════════════════════════════════════════════════════════
# CELL 1 — Install dependencies
# ════════════════════════════════════════════════════════════════════════
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'plotly==5.18.0', 'openpyxl', 'kaleido==0.2.1', 'pandas',
                '--quiet'], check=False)
print('✔  Dependencies ready')

✔  Dependencies ready


In [2]:
# ════════════════════════════════════════════════════════════════════════
# CELL 2 — Load & parse Excel files
# Place the two xlsx files in the same directory as this notebook:
#   JWLG-OVTP_Hydrographs.xlsx
#   JWLG-PIPG_Hydrographs.xlsx
# ════════════════════════════════════════════════════════════════════════
import openpyxl, datetime, math
import numpy as np
import pandas as pd

BASE = datetime.datetime(2026, 1, 1, 0, 0, 0)   # Simulation t = 0

def load_sheet(ws, step_s, col_map):
    """Load a worksheet into a list of dicts with time in hours."""
    data = []
    for i, row in enumerate(ws.iter_rows(min_row=2, values_only=True)):
        hr = i * step_s / 3600
        entry = {'hr': round(hr, 6)}
        for key, col in col_map.items():
            val = row[col]
            entry[key] = float(val) if val is not None else 0.0
        data.append(entry)
    return data

print('Loading OVTP data ...')
wb_o = openpyxl.load_workbook('JWLG-OVTP Hydrographs.xlsx')
o_main  = load_sheet(wb_o['Breach OVTP'],   5, {'hw':2,'tw':3,'total_q':4,'weir_q':5,'breach_q':6})
o_width = load_sheet(wb_o['Breach Width'],  5, {'width':2})
o_vel   = load_sheet(wb_o['Breach Velocity'],5,{'velocity':2})
o_ds    = load_sheet(wb_o['DS'],            5, {'ds_elev':2,'ds_q':3})
print(f'  ✔  OVTP: {len(o_main):,} rows | {o_main[-1]["hr"]:.2f} hr')

print('Loading PIPG data ...')
wb_p = openpyxl.load_workbook('JWLG-PIPG Hydrographs.xlsx')
p_main  = load_sheet(wb_p['Brech_Pipg'], 1, {'hw':2,'tw':3,'total_q':4,'weir_q':5,'breach_q':6})
p_width = load_sheet(wb_p['Width'],      1, {'width':2})
p_vel   = load_sheet(wb_p['Velocity'],   1, {'velocity':2})
p_ds    = load_sheet(wb_p['DS'],         1, {'ds_elev':2,'ds_q':3})
print(f'  ✔  PIPG: {len(p_main):,} rows | {p_main[-1]["hr"]:.2f} hr')

# ── Subsample to ~30-second resolution for smooth plots ───────────────
def ss(lst, n):
    return lst[::n]

# OVTP: 5s * 6 = 30s resolution
om = ss(o_main, 6);   ow = ss(o_width, 6)
ov = ss(o_vel,  6);   od = ss(o_ds,   6)
# PIPG: 1s * 30 = 30s resolution
pm = ss(p_main, 30);  pw = ss(p_width, 30)
pv = ss(p_vel,  30);  pds = ss(p_ds,  30)

print(f'  Subsampled OVTP: {len(om):,} pts | PIPG: {len(pm):,} pts')

Loading OVTP data ...
  ✔  OVTP: 51,841 rows | 72.00 hr
Loading PIPG data ...
  ✔  PIPG: 259,201 rows | 72.00 hr
  Subsampled OVTP: 8,641 pts | PIPG: 8,641 pts


In [3]:
# ════════════════════════════════════════════════════════════════════════
# CELL 3 — Compute all key statistics + FLOOD ARRIVAL TIMES
# ════════════════════════════════════════════════════════════════════════
def hm(hr):
    """Convert decimal hours to '##h ##m ##s' string."""
    total_sec = hr * 3600
    h = int(total_sec // 3600)
    m = int((total_sec % 3600) // 60)
    sc = int(total_sec % 60)
    return f'{h}h {m:02d}m {sc:02d}s'

def hm_short(hr):
    """Short version for plot labels."""
    h = int(hr); m = int((hr % 1) * 60)
    return f'{h}h {m:02d}m'

# ─── OVTP ──────────────────────────────────────────────────────────────
O_PEAK      = max(o_main,  key=lambda x: x['total_q'])
O_MAX_HW    = max(o_main,  key=lambda x: x['hw'])
O_MAX_BW    = max(o_width, key=lambda x: x['width'])
O_MAX_VEL   = max(o_vel,   key=lambda x: x['velocity'])
O_MAX_DS    = max(o_ds,    key=lambda x: x['ds_q'])
O_INIT      = next(r for r in o_main  if r['total_q'] > 0)
O_DS_INIT   = next(r for r in o_ds    if r['ds_q']    > 0)

# Note: OVTP velocity column carries identical values to breach width
# (both columns share same model output in this dataset)
OVTP_TRAVEL_HR   = O_MAX_DS['hr'] - O_INIT['hr']       # breach start → DS peak
OVTP_LAG_HR      = O_MAX_DS['hr'] - O_PEAK['hr']        # US peak → DS peak

# ─── PIPG ──────────────────────────────────────────────────────────────
P_PEAK      = max(p_main,  key=lambda x: x['total_q'])
P_MAX_HW    = max(p_main,  key=lambda x: x['hw'])
P_MAX_BW    = max(p_width, key=lambda x: x['width'])
P_MAX_VEL   = max(p_vel,   key=lambda x: x['velocity'])
P_MAX_DS    = max(p_ds,    key=lambda x: x['ds_q'])
P_INIT      = next(r for r in p_main if r['breach_q'] and r['breach_q'] > 0)
P_DS_INIT   = next(r for r in p_ds   if r['ds_q'] > 0)

PIPG_TRAVEL_HR   = P_MAX_DS['hr'] - P_INIT['hr']        # pipe start → DS peak
PIPG_LAG_HR      = P_MAX_DS['hr'] - P_PEAK['hr']        # US peak → DS peak

# ─── Print summary ────────────────────────────────────────────────────
SEP = '='*72
print(SEP)
print('  JAWALGAON DAM — DBA KEY STATISTICS')
print(SEP)

print('\n┌─ OVERTOPPING (OVTP) ─────────────────────────────────────────────')
print(f'│  Peak Total Discharge     : {O_PEAK["total_q"]:>10.2f}  m³/s  @ {hm(O_PEAK["hr"])}')
print(f'│  Max Reservoir HW Elev    : {O_MAX_HW["hw"]:>10.3f}  m     @ {hm(O_MAX_HW["hr"])}')
print(f'│  Max Breach Width         : {O_MAX_BW["width"]:>10.2f}  m     @ {hm(O_MAX_BW["hr"])}')
print(f'│  Max Breach Velocity      : {O_MAX_VEL["velocity"]:>10.3f}  m/s   @ {hm(O_MAX_VEL["hr"])}')
print(f'│  Max Downstream Q         : {O_MAX_DS["ds_q"]:>10.2f}  m³/s  @ {hm(O_MAX_DS["hr"])}')
print(f'│  Breach Initiation        :              {hm(O_INIT["hr"])}')
print(f'│  DS Flood Arrival (first) :              {hm(O_DS_INIT["hr"])}')
print(f'├─ FLOOD ARRIVAL TIMES ────────────────────────────────────────────')
print(f'│  Breach Start → DS Peak   :  {OVTP_TRAVEL_HR:.4f} hr  =  {hm(OVTP_TRAVEL_HR)}  ← TRAVEL TIME')
print(f'│  US Peak      → DS Peak   :  {OVTP_LAG_HR:.4f} hr  =  {hm(OVTP_LAG_HR)}  ← ATTENUATION LAG')
print('└──────────────────────────────────────────────────────────────────')

print('\n┌─ PIPING (PIPG) ───────────────────────────────────────────────────')
print(f'│  Peak Total Discharge     : {P_PEAK["total_q"]:>10.2f}  m³/s  @ {hm(P_PEAK["hr"])}')
print(f'│  Max Reservoir HW Elev    : {P_MAX_HW["hw"]:>10.3f}  m     @ {hm(P_MAX_HW["hr"])}')
print(f'│  Max Breach Width         : {P_MAX_BW["width"]:>10.2f}  m     @ {hm(P_MAX_BW["hr"])}')
print(f'│  Max Breach Velocity      : {P_MAX_VEL["velocity"]:>10.3f}  m/s   @ {hm(P_MAX_VEL["hr"])}')
print(f'│  Max Downstream Q         : {P_MAX_DS["ds_q"]:>10.2f}  m³/s  @ {hm(P_MAX_DS["hr"])}')
print(f'│  Piping Initiation        :              {hm(P_INIT["hr"])}')
print(f'│  DS Flood Arrival (first) :              {hm(P_DS_INIT["hr"])}')
print(f'├─ FLOOD ARRIVAL TIMES ────────────────────────────────────────────')
print(f'│  Pipe Start  → DS Peak    :  {PIPG_TRAVEL_HR:.4f} hr  =  {hm(PIPG_TRAVEL_HR)}  ← TRAVEL TIME')
print(f'│  US Peak     → DS Peak    :  {PIPG_LAG_HR:.4f} hr  =  {hm(PIPG_LAG_HR)}  ← ATTENUATION LAG')
print('└──────────────────────────────────────────────────────────────────')

print(f'\n┌─ COMPARISON ──────────────────────────────────────────────────────')
print(f'│  Parameter                        OVTP              PIPG')
print(f'│  Peak Discharge              {O_PEAK["total_q"]:>9.2f} m³/s   {P_PEAK["total_q"]:>9.2f} m³/s')
print(f'│  Time to Peak                 {hm_short(O_PEAK["hr"]):>12}      {hm_short(P_PEAK["hr"]):>12}')
print(f'│  Max DS Flood Q              {O_MAX_DS["ds_q"]:>9.2f} m³/s   {P_MAX_DS["ds_q"]:>9.2f} m³/s')
print(f'│  Flood Travel Time (hrs+min)  {hm_short(OVTP_TRAVEL_HR):>12}      {hm_short(PIPG_TRAVEL_HR):>12}')
print(f'│  US↔DS Peak Attenuation Lag   {hm_short(OVTP_LAG_HR):>12}      {hm_short(PIPG_LAG_HR):>12}')
print(f'│  Max Breach Width            {O_MAX_BW["width"]:>9.2f} m       {P_MAX_BW["width"]:>9.2f} m')
print('└──────────────────────────────────────────────────────────────────')

  JAWALGAON DAM — DBA KEY STATISTICS

┌─ OVERTOPPING (OVTP) ─────────────────────────────────────────────
│  Peak Total Discharge     :    4128.22  m³/s  @ 19h 34m 40s
│  Max Reservoir HW Elev    :    508.640  m     @ 16h 05m 49s
│  Max Breach Width         :     117.00  m     @ 20h 39m 15s
│  Max Breach Velocity      :      4.570  m/s   @ 17h 43m 14s
│  Max Downstream Q         :    3808.01  m³/s  @ 23h 15m 20s
│  Breach Initiation        :              14h 08m 25s
│  DS Flood Arrival (first) :              20h 08m 15s
├─ FLOOD ARRIVAL TIMES ────────────────────────────────────────────
│  Breach Start → DS Peak   :  9.1153 hr  =  9h 06m 55s  ← TRAVEL TIME
│  US Peak      → DS Peak   :  3.6778 hr  =  3h 40m 40s  ← ATTENUATION LAG
└──────────────────────────────────────────────────────────────────

┌─ PIPING (PIPG) ───────────────────────────────────────────────────
│  Peak Total Discharge     :    1567.10  m³/s  @ 2h 50m 26s
│  Max Reservoir HW Elev    :    503.070  m     @ 0h 00m 01s


In [4]:
# ════════════════════════════════════════════════════════════════════════
# CELL 4 — Styling helpers (engineering graph-paper style)
# ════════════════════════════════════════════════════════════════════════
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Colour palette ────────────────────────────────────────────────────
C_OVTP  = '#1565C0'   # OVTP blue
C_PIPG  = '#B71C1C'   # PIPG red
C_TW    = '#E65100'   # tail-water orange
C_DS    = '#2E7D32'   # downstream green
C_BW    = '#6A1B9A'   # breach width violet
C_VEL   = '#00695C'   # velocity teal
C_BR    = '#880E4F'   # breach Q deep pink

GRID    = 'rgba(155,155,155,0.50)'
GRID_MI = 'rgba(200,200,210,0.28)'
BG      = '#F8FAFB'
PAPER   = '#EBEBEB'

def _base_layout(title, w=1300, h=560):
    return dict(
        title=dict(text=title, font=dict(family='Arial', size=13, color='#111'),
                   x=0.5, xanchor='center', y=0.97),
        paper_bgcolor=PAPER,
        plot_bgcolor=BG,
        width=w, height=h,
        margin=dict(l=85, r=55, t=90, b=115),
        legend=dict(
            bgcolor='rgba(255,255,255,0.92)',
            bordercolor='#444', borderwidth=1.5,
            font=dict(family='Arial', size=11.5),
            x=0.99, y=0.99, xanchor='right', yanchor='top'
        ),
        font=dict(family='Arial')
    )

def apply_eng_axes(fig, x_dtick=6, y_dtick=None,
                   xtitle='Elapsed Time (hr)', ytitle='',
                   xrange=None, yrange=None):
    ax_common = dict(
        showgrid=True, gridcolor=GRID, gridwidth=1,
        zeroline=True, zerolinecolor='#2a2a2a', zerolinewidth=1.5,
        linecolor='#2a2a2a', linewidth=1.8,
        mirror=True, ticks='outside', ticklen=7, tickwidth=1.5,
        tickfont=dict(family='Courier New', size=9.5),
    )
    xargs = {**ax_common, 'title_text': xtitle,
             'dtick': x_dtick,
             'minor': dict(showgrid=True, gridcolor=GRID_MI, gridwidth=0.5, dtick=x_dtick/6)}
    yargs = {**ax_common, 'title_text': ytitle,
             'minor': dict(showgrid=True, gridcolor=GRID_MI, gridwidth=0.5)}
    if y_dtick: yargs['dtick'] = y_dtick
    if xrange:  xargs['range'] = xrange
    if yrange:  yargs['range'] = yrange
    fig.update_xaxes(**xargs)
    fig.update_yaxes(**yargs)
    return fig

def add_6hr_lines(fig, max_h=73):
    for h in range(6, max_h, 6):
        fig.add_vline(x=h, line_dash='dot',
                      line_color='rgba(90,90,90,0.30)', line_width=1)

def title_block(fig, line1, line2=''):
    txt = f'<b>{line1}</b>' + (f'<br>{line2}' if line2 else '')
    fig.add_annotation(
        text=txt, xref='paper', yref='paper', x=0.5, y=-0.16,
        showarrow=False,
        font=dict(family='Courier New', size=8.5, color='#333'),
        bgcolor='rgba(218,218,218,0.92)', bordercolor='#666',
        borderwidth=1, align='center')

def peak_arrow(fig, t, q, color, label):
    fig.add_annotation(
        x=t, y=q, text=label,
        showarrow=True, arrowhead=2, arrowwidth=1.5,
        arrowcolor=color, ax=65, ay=-52,
        font=dict(size=10, color=color, family='Arial'),
        bgcolor='rgba(255,255,255,0.88)',
        bordercolor=color, borderwidth=1)

def arrival_arrow(fig, t, q, color, label, ax=-70, ay=-40):
    fig.add_annotation(
        x=t, y=q, text=label,
        showarrow=True, arrowhead=3, arrowwidth=1.5,
        arrowcolor=color, ax=ax, ay=ay,
        font=dict(size=9.5, color=color, family='Arial'),
        bgcolor='rgba(255,255,255,0.88)',
        bordercolor=color, borderwidth=1)

def scale_box(fig, x_pos, y_pos, x_span_hr, y_span, unit_x='hr', unit_y='m³/s'):
    """Draw a simple scale bar box on the plot."""
    fig.add_shape(type='rect',
        x0=x_pos, x1=x_pos+x_span_hr, y0=y_pos, y1=y_pos+y_span,
        line=dict(color='#888', width=1.5),
        fillcolor='rgba(240,240,240,0.7)')
    fig.add_annotation(x=x_pos+x_span_hr/2, y=y_pos+y_span*1.12,
        text=f'← {x_span_hr} {unit_x} →', showarrow=False,
        font=dict(size=8, color='#555', family='Arial'))
    fig.add_annotation(x=x_pos+x_span_hr+0.8, y=y_pos+y_span/2,
        text=f'{y_span} {unit_y}', showarrow=False, textangle=-90,
        font=dict(size=8, color='#555', family='Arial'))

print('✔  Styling helpers defined')

✔  Styling helpers defined


---
## 1. OVERTOPPING (OVTP) FAILURE

In [5]:
# ════════════════════════════════════════════════════════════════════════
# PLOT 1.1 — OVTP: HW, TW Elevation & Total Flow
# ════════════════════════════════════════════════════════════════════════
fig11 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[
        '(a) Reservoir HW & Tailwater TW Elevation (m)',
        '(b) Total Discharge & Breach Flow (m³/s)'
    ],
    vertical_spacing=0.10, row_heights=[0.45, 0.55]
)

# ─ HW elevation ────────────────────────────────────────────────────────
fig11.add_trace(go.Scatter(
    x=[r['hr'] for r in om], y=[r['hw'] for r in om],
    name='HW — Reservoir (Upstream)', mode='lines',
    line=dict(color=C_OVTP, width=2.5)
), row=1, col=1)

# ─ TW elevation ────────────────────────────────────────────────────────
fig11.add_trace(go.Scatter(
    x=[r['hr'] for r in om], y=[r['tw'] for r in om],
    name='TW — Tailwater (Downstream)', mode='lines',
    line=dict(color=C_TW, width=2, dash='dash')
), row=1, col=1)

# FRL reference line
fig11.add_hline(y=503.07, line_dash='dot', line_color='#26A69A', line_width=1.5,
                annotation_text='FRL = 503.07 m', annotation_position='top right',
                annotation_font=dict(size=9, color='#26A69A'), row=1, col=1)

# HW peak annotation
fig11.add_annotation(
    x=O_MAX_HW['hr'], y=O_MAX_HW['hw'],
    text=f"<b>Max HW = {O_MAX_HW['hw']:.3f} m<br>@ {hm_short(O_MAX_HW['hr'])}</b>",
    showarrow=True, arrowhead=2, arrowcolor=C_OVTP, ax=-60, ay=-45,
    font=dict(size=10, color=C_OVTP),
    bgcolor='rgba(255,255,255,0.88)', bordercolor=C_OVTP, borderwidth=1,
    row=1, col=1)

# ─ Total discharge ─────────────────────────────────────────────────────
fig11.add_trace(go.Scatter(
    x=[r['hr'] for r in om], y=[r['total_q'] for r in om],
    name='Total Discharge Q (m³/s)', mode='lines',
    line=dict(color=C_OVTP, width=2.5),
    fill='tozeroy', fillcolor='rgba(21,101,192,0.10)'
), row=2, col=1)

# ─ Breach flow component ───────────────────────────────────────────────
fig11.add_trace(go.Scatter(
    x=[r['hr'] for r in om],
    y=[r['breach_q'] if r['breach_q'] else 0 for r in om],
    name='Breach Flow Component', mode='lines',
    line=dict(color=C_BR, width=1.8, dash='dash')
), row=2, col=1)

# Peak Q annotation
pk_i = max(range(len(om)), key=lambda i: om[i]['total_q'])
peak_arrow(fig11, om[pk_i]['hr'], om[pk_i]['total_q'], C_OVTP,
           f"<b>Peak Q = {om[pk_i]['total_q']:.0f} m³/s<br>@ {hm_short(om[pk_i]['hr'])}</b>")

# Breach initiation vline
fig11.add_vline(x=O_INIT['hr'], line_dash='dashdot', line_color='#F57F17', line_width=1.5)
fig11.add_annotation(x=O_INIT['hr'], y=0.05, yref='paper',
    text=f"Breach init<br>{hm_short(O_INIT['hr'])}",
    showarrow=False, font=dict(size=8.5, color='#F57F17'),
    bgcolor='rgba(255,249,196,0.88)', bordercolor='#F57F17', borderwidth=1)

add_6hr_lines(fig11)
fig11.update_layout(**_base_layout(
    '<b>JAWALGAON DAM — OVTP FAILURE (1.1) : HW / TW ELEVATION & FLOW</b><br>'
    '<span style="font-size:11px">DBA – Standard Project Flood | 72-Hr Simulation | '
    'REF: DBA-JWLG-OVTP-1.1</span>',
    w=1300, h=640
))
fig11.update_xaxes(
    title_text='Elapsed Time (hr from breach initiation)',
    dtick=6, range=[-0.5, 73],
    minor=dict(showgrid=True, gridcolor=GRID_MI, dtick=1),
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5),
    row=2, col=1)
fig11.update_yaxes(
    title_text='Elevation (m)', row=1, col=1,
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
fig11.update_yaxes(
    title_text='Discharge Q (m³/s)', row=2, col=1, dtick=500,
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
title_block(fig11,
    f'DAM: Jawalgaon Dam | FAILURE: Overtopping (OVTP) | '
    f'Peak Q = {O_PEAK["total_q"]:.0f} m³/s @ {hm_short(O_PEAK["hr"])} | '
    f'Max HW = {O_MAX_HW["hw"]:.3f} m @ {hm_short(O_MAX_HW["hr"])}',
    f'Breach Init: {hm(O_INIT["hr"])} | SCALE: H:1cm=6hr  V:1cm=500m³/s | REF: DBA-JWLG-OVTP-1.1')
fig11.write_html('OVTP_1.1_HW_TW_Flow.html')
fig11.show()
print('✔  Fig 1.1 saved → OVTP_1.1_HW_TW_Flow.html')

✔  Fig 1.1 saved → OVTP_1.1_HW_TW_Flow.html


In [6]:
# ════════════════════════════════════════════════════════════════════════
# PLOT 1.2 — OVTP: Breach Width
# ════════════════════════════════════════════════════════════════════════
fig12 = go.Figure()

fig12.add_trace(go.Scatter(
    x=[r['hr'] for r in ow], y=[r['width'] for r in ow],
    name='Breach Width (m)', mode='lines',
    line=dict(color=C_BW, width=2.8),
    fill='tozeroy', fillcolor='rgba(106,27,154,0.10)'
))

# Max breach width annotation
fig12.add_annotation(
    x=O_MAX_BW['hr'], y=O_MAX_BW['width'],
    text=f"<b>Max Width = {O_MAX_BW['width']:.1f} m<br>@ {hm_short(O_MAX_BW['hr'])}</b>",
    showarrow=True, arrowhead=2, arrowcolor=C_BW, ax=-70, ay=-45,
    font=dict(size=10.5, color=C_BW),
    bgcolor='rgba(255,255,255,0.88)', bordercolor=C_BW, borderwidth=1)

# Breach initiation vline
fig12.add_vline(x=O_INIT['hr'], line_dash='dashdot',
                line_color='#F57F17', line_width=1.5,
                annotation_text=f"Breach init {hm_short(O_INIT['hr'])}",
                annotation_position='top right',
                annotation_font=dict(size=9, color='#F57F17'))

# Phase labels
fig12.add_annotation(x=17, y=60,
    text='▶ Active Breach<br>   Widening Phase',
    showarrow=False, font=dict(size=9, color=C_BW),
    bgcolor='rgba(237,231,246,0.8)', bordercolor=C_BW, borderwidth=1)
fig12.add_annotation(x=35, y=40,
    text='Full Width<br>117 m (stable)',
    showarrow=False, font=dict(size=9, color='#555'),
    bgcolor='rgba(245,245,245,0.85)', bordercolor='#999', borderwidth=1)

add_6hr_lines(fig12)
fig12.update_layout(**_base_layout(
    '<b>JAWALGAON DAM — OVTP FAILURE (1.2) : BREACH WIDTH DEVELOPMENT</b><br>'
    '<span style="font-size:11px">Progressive Embankment Erosion & Breach Formation | '
    'REF: DBA-JWLG-OVTP-1.2</span>',
    w=1300, h=520
))
fig12.update_xaxes(
    title_text='Elapsed Time (hr)', dtick=6, range=[-0.5, 73],
    minor=dict(showgrid=True, gridcolor=GRID_MI, dtick=1),
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
fig12.update_yaxes(
    title_text='Breach Width (m)', dtick=20, range=[-2, 130],
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5),
    minor=dict(showgrid=True, gridcolor=GRID_MI, dtick=5))
title_block(fig12,
    f'DAM: Jawalgaon Dam | FAILURE: Overtopping (OVTP) | '
    f'Max Breach Width = {O_MAX_BW["width"]:.1f} m @ {hm_short(O_MAX_BW["hr"])} | '
    f'Width Stable from {hm_short(O_MAX_BW["hr"])} onwards',
    'SCALE: H:1cm=6hr  V:1cm=20m | REF: DBA-JWLG-OVTP-1.2')
fig12.write_html('OVTP_1.2_Breach_Width.html')
fig12.show()
print('✔  Fig 1.2 saved → OVTP_1.2_Breach_Width.html')

✔  Fig 1.2 saved → OVTP_1.2_Breach_Width.html


In [7]:
# ════════════════════════════════════════════════════════════════════════
# PLOT 1.3 — OVTP: Breach Velocity
# Note: In this dataset the OVTP Breach Velocity column contains the
# same values as Breach Width (model output artefact). Both are plotted
# with a note; the velocity axis is labelled accordingly.
# ════════════════════════════════════════════════════════════════════════
fig13 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[
        '(a) Breach Velocity (m/s) — from model output',
        '(b) Breach Width vs Velocity Overlay (both axes)'
    ],
    vertical_spacing=0.12, row_heights=[0.50, 0.50],
    specs=[[{'secondary_y': False}], [{'secondary_y': True}]]
)

# Panel a — velocity
fig13.add_trace(go.Scatter(
    x=[r['hr'] for r in ov], y=[r['velocity'] for r in ov],
    name='Breach Velocity (m/s)', mode='lines',
    line=dict(color=C_VEL, width=2.5),
    fill='tozeroy', fillcolor='rgba(0,105,92,0.10)'
), row=1, col=1)

# Max velocity annotation
fig13.add_annotation(
    x=O_MAX_VEL['hr'], y=O_MAX_VEL['velocity'],
    text=f"<b>Max = {O_MAX_VEL['velocity']:.2f}<br>@ {hm_short(O_MAX_VEL['hr'])}</b>",
    showarrow=True, arrowhead=2, arrowcolor=C_VEL, ax=-70, ay=-45,
    font=dict(size=10, color=C_VEL),
    bgcolor='rgba(255,255,255,0.88)', bordercolor=C_VEL, borderwidth=1,
    row=1, col=1)

# Panel b — overlay width + velocity on dual axis
fig13.add_trace(go.Scatter(
    x=[r['hr'] for r in ow], y=[r['width'] for r in ow],
    name='Breach Width (m) [Left axis]', mode='lines',
    line=dict(color=C_BW, width=2.2)
), row=2, col=1, secondary_y=False)

fig13.add_trace(go.Scatter(
    x=[r['hr'] for r in ov], y=[r['velocity'] for r in ov],
    name='Breach Velocity (m/s) [Right axis]', mode='lines',
    line=dict(color=C_VEL, width=2, dash='dot')
), row=2, col=1, secondary_y=True)

add_6hr_lines(fig13)
fig13.update_layout(**_base_layout(
    '<b>JAWALGAON DAM — OVTP FAILURE (1.3) : BREACH VELOCITY</b><br>'
    '<span style="font-size:11px">Breach Flow Velocity at Dam Section | '
    'REF: DBA-JWLG-OVTP-1.3</span>',
    w=1300, h=640
))
fig13.update_xaxes(
    title_text='Elapsed Time (hr)', dtick=6, range=[-0.5, 73],
    minor=dict(showgrid=True, gridcolor=GRID_MI, dtick=1),
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5),
    row=2, col=1)
fig13.update_yaxes(
    title_text='Velocity (m/s)', row=1, col=1,
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
fig13.update_yaxes(title_text='Breach Width (m)', row=2, col=1, secondary_y=False)
fig13.update_yaxes(title_text='Velocity (m/s)', row=2, col=1, secondary_y=True)
title_block(fig13,
    f'DAM: Jawalgaon Dam | FAILURE: Overtopping (OVTP) | '
    f'Max Breach Velocity = {O_MAX_VEL["velocity"]:.3f} m/s @ {hm_short(O_MAX_VEL["hr"])}',
    'REF: DBA-JWLG-OVTP-1.3')
fig13.write_html('OVTP_1.3_Breach_Velocity.html')
fig13.show()
print('✔  Fig 1.3 saved → OVTP_1.3_Breach_Velocity.html')

✔  Fig 1.3 saved → OVTP_1.3_Breach_Velocity.html


In [8]:
# ════════════════════════════════════════════════════════════════════════
# PLOT 1.4 — OVTP: Downstream Hydrograph + Flood Arrival Time
# ════════════════════════════════════════════════════════════════════════
fig14 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[
        '(a) Downstream Flood Discharge (m³/s) — with Flood Arrival Markers',
        '(b) Downstream Stage / Water Surface Elevation (m)'
    ],
    vertical_spacing=0.10, row_heights=[0.60, 0.40]
)

# Downstream flow
fig14.add_trace(go.Scatter(
    x=[r['hr'] for r in od], y=[r['ds_q'] for r in od],
    name='Downstream Flood Q (m³/s)', mode='lines',
    line=dict(color=C_DS, width=2.8),
    fill='tozeroy', fillcolor='rgba(46,125,50,0.12)'
), row=1, col=1)

# Upstream flow on same panel for comparison
fig14.add_trace(go.Scatter(
    x=[r['hr'] for r in om], y=[r['total_q'] for r in om],
    name='Upstream Breach Q (m³/s) [reference]', mode='lines',
    line=dict(color=C_OVTP, width=1.8, dash='dot'),
    opacity=0.65
), row=1, col=1)

# Peak DS annotation
ds_i = max(range(len(od)), key=lambda i: od[i]['ds_q'])
fig14.add_annotation(
    x=od[ds_i]['hr'], y=od[ds_i]['ds_q'],
    text=f"<b>DS Peak Q = {od[ds_i]['ds_q']:.0f} m³/s<br>@ {hm_short(od[ds_i]['hr'])}</b>",
    showarrow=True, arrowhead=2, arrowcolor=C_DS, ax=-75, ay=-48,
    font=dict(size=10.5, color=C_DS),
    bgcolor='rgba(255,255,255,0.88)', bordercolor=C_DS, borderwidth=1,
    row=1, col=1)

# ── FLOOD ARRIVAL ANNOTATIONS ──────────────────────────────────────────
# First DS flow arrival
fig14.add_vline(x=O_DS_INIT['hr'], line_dash='dashdot',
                line_color='#FF8F00', line_width=1.8)
fig14.add_annotation(
    x=O_DS_INIT['hr'], y=0.62, yref='paper',
    text=f"<b>▼ Flood Arrives DS</b><br>{hm_short(O_DS_INIT['hr'])}",
    showarrow=False, font=dict(size=9, color='#FF8F00'),
    bgcolor='rgba(255,248,225,0.90)', bordercolor='#FF8F00', borderwidth=1)

# Breach initiation
fig14.add_vline(x=O_INIT['hr'], line_dash='dashdot',
                line_color='#F57F17', line_width=1.5)
fig14.add_annotation(
    x=O_INIT['hr'], y=0.90, yref='paper',
    text=f"<b>▲ Breach Starts</b><br>{hm_short(O_INIT['hr'])}",
    showarrow=False, font=dict(size=9, color='#F57F17'),
    bgcolor='rgba(255,253,231,0.90)', bordercolor='#F57F17', borderwidth=1)

# Travel time arrow (breach start → DS peak)
fig14.add_annotation(
    x=O_INIT['hr'], y=od[ds_i]['ds_q']*0.35,
    ax=O_MAX_DS['hr'], ay=od[ds_i]['ds_q']*0.35,
    xref='x', yref='y', axref='x', ayref='y',
    text=f"  Travel Time: {hm(OVTP_TRAVEL_HR)}",
    showarrow=True, arrowhead=2, arrowside='end+start',
    arrowcolor='#5C6BC0', arrowwidth=2,
    font=dict(size=10, color='#3949AB', family='Arial'),
    bgcolor='rgba(232,234,246,0.92)', bordercolor='#5C6BC0', borderwidth=1,
    row=1, col=1)

# Attenuation lag annotation
fig14.add_annotation(
    x=O_PEAK['hr'], y=od[ds_i]['ds_q']*0.70,
    ax=O_MAX_DS['hr'], ay=od[ds_i]['ds_q']*0.70,
    xref='x', yref='y', axref='x', ayref='y',
    text=f"  Peak Lag: {hm(OVTP_LAG_HR)}",
    showarrow=True, arrowhead=2, arrowside='end+start',
    arrowcolor='#00838F', arrowwidth=2,
    font=dict(size=10, color='#006064', family='Arial'),
    bgcolor='rgba(224,247,250,0.92)', bordercolor='#00838F', borderwidth=1,
    row=1, col=1)

# Downstream elevation
fig14.add_trace(go.Scatter(
    x=[r['hr'] for r in od], y=[r['ds_elev'] for r in od],
    name='DS Water Surface Elevation (m)', mode='lines',
    line=dict(color='#0277BD', width=2.2),
    fill='tozeroy', fillcolor='rgba(2,119,189,0.08)'
), row=2, col=1)

add_6hr_lines(fig14)
fig14.update_layout(**_base_layout(
    '<b>JAWALGAON DAM — OVTP FAILURE (1.4) : DOWNSTREAM FLOOD HYDROGRAPH</b><br>'
    f'<span style="font-size:11px">Flood Travel Time = {hm(OVTP_TRAVEL_HR)} | '
    f'Peak Attenuation Lag = {hm(OVTP_LAG_HR)} | REF: DBA-JWLG-OVTP-1.4</span>',
    w=1300, h=660
))
fig14.update_xaxes(
    title_text='Elapsed Time (hr)', dtick=6, range=[-0.5, 73],
    minor=dict(showgrid=True, gridcolor=GRID_MI, dtick=1),
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5),
    row=2, col=1)
fig14.update_yaxes(
    title_text='Discharge Q (m³/s)', dtick=500, row=1, col=1,
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
fig14.update_yaxes(
    title_text='Elevation (m)', row=2, col=1,
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
title_block(fig14,
    f'DAM: Jawalgaon Dam | DS Peak Q = {O_MAX_DS["ds_q"]:.0f} m³/s @ {hm_short(O_MAX_DS["hr"])} | '
    f'Flood Travel Time = {hm(OVTP_TRAVEL_HR)} | Peak Lag = {hm(OVTP_LAG_HR)}',
    'SCALE: H:1cm=6hr  V:1cm=500m³/s | REF: DBA-JWLG-OVTP-1.4')
fig14.write_html('OVTP_1.4_Downstream_Hydrograph.html')
fig14.show()
print('✔  Fig 1.4 saved → OVTP_1.4_Downstream_Hydrograph.html')

✔  Fig 1.4 saved → OVTP_1.4_Downstream_Hydrograph.html


---
## 2. PIPING (PIPG) FAILURE

In [9]:
# ════════════════════════════════════════════════════════════════════════
# PLOT 2.1 — PIPG: HW, TW Elevation & Total Flow
# ════════════════════════════════════════════════════════════════════════
fig21 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[
        '(a) Reservoir HW & Tailwater TW Elevation (m)',
        '(b) Total Discharge & Breach (Piping) Flow (m³/s)'
    ],
    vertical_spacing=0.10, row_heights=[0.45, 0.55]
)

# HW
fig21.add_trace(go.Scatter(
    x=[r['hr'] for r in pm], y=[r['hw'] for r in pm],
    name='HW — Reservoir (Upstream)', mode='lines',
    line=dict(color=C_PIPG, width=2.5)
), row=1, col=1)

# TW
fig21.add_trace(go.Scatter(
    x=[r['hr'] for r in pm], y=[r['tw'] for r in pm],
    name='TW — Tailwater (Downstream)', mode='lines',
    line=dict(color=C_TW, width=2, dash='dash')
), row=1, col=1)

# FRL line
fig21.add_hline(y=503.07, line_dash='dot', line_color='#26A69A', line_width=1.5,
                annotation_text='FRL = 503.07 m', annotation_position='top right',
                annotation_font=dict(size=9, color='#26A69A'), row=1, col=1)

# Total discharge
fig21.add_trace(go.Scatter(
    x=[r['hr'] for r in pm], y=[r['total_q'] for r in pm],
    name='Total Discharge Q (m³/s)', mode='lines',
    line=dict(color=C_PIPG, width=2.5),
    fill='tozeroy', fillcolor='rgba(183,28,28,0.10)'
), row=2, col=1)

# Breach (piping) Q
fig21.add_trace(go.Scatter(
    x=[r['hr'] for r in pm], y=[r['breach_q'] for r in pm],
    name='Breach (Piping) Flow Component', mode='lines',
    line=dict(color=C_BR, width=1.8, dash='dash')
), row=2, col=1)

# Peak Q annotation
ppk_i = max(range(len(pm)), key=lambda i: pm[i]['total_q'])
fig21.add_annotation(
    x=pm[ppk_i]['hr'], y=pm[ppk_i]['total_q'],
    text=f"<b>Peak Q = {pm[ppk_i]['total_q']:.0f} m³/s<br>@ {hm_short(pm[ppk_i]['hr'])}</b>",
    showarrow=True, arrowhead=2, arrowcolor=C_PIPG, ax=70, ay=-50,
    font=dict(size=10, color=C_PIPG),
    bgcolor='rgba(255,255,255,0.88)', bordercolor=C_PIPG, borderwidth=1,
    row=2, col=1)

# Piping initiation
fig21.add_vline(x=P_INIT['hr'], line_dash='dashdot',
                line_color='#F57F17', line_width=1.5)
fig21.add_annotation(
    x=P_INIT['hr'], y=0.05, yref='paper',
    text=f"Piping init<br>{hm_short(P_INIT['hr'])}",
    showarrow=False, font=dict(size=8.5, color='#F57F17'),
    bgcolor='rgba(255,249,196,0.88)', bordercolor='#F57F17', borderwidth=1)

add_6hr_lines(fig21)
fig21.update_layout(**_base_layout(
    '<b>JAWALGAON DAM — PIPG FAILURE (2.1) : HW / TW ELEVATION & FLOW</b><br>'
    '<span style="font-size:11px">DBA – Standard Project Flood | 72-Hr Simulation | '
    'REF: DBA-JWLG-PIPG-2.1</span>',
    w=1300, h=640
))
fig21.update_xaxes(
    title_text='Elapsed Time (hr from piping initiation)',
    dtick=6, range=[-0.5, 73],
    minor=dict(showgrid=True, gridcolor=GRID_MI, dtick=1),
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5), row=2, col=1)
fig21.update_yaxes(
    title_text='Elevation (m)', row=1, col=1,
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
fig21.update_yaxes(
    title_text='Discharge Q (m³/s)', dtick=200, row=2, col=1,
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
title_block(fig21,
    f'DAM: Jawalgaon Dam | FAILURE: Piping (PIPG) | '
    f'Peak Q = {P_PEAK["total_q"]:.0f} m³/s @ {hm_short(P_PEAK["hr"])} | '
    f'Max HW = {P_MAX_HW["hw"]:.3f} m @ {hm_short(P_MAX_HW["hr"])}',
    f'Piping Init: {hm(P_INIT["hr"])} | SCALE: H:1cm=6hr  V:1cm=200m³/s | REF: DBA-JWLG-PIPG-2.1')
fig21.write_html('PIPG_2.1_HW_TW_Flow.html')
fig21.show()
print('✔  Fig 2.1 saved → PIPG_2.1_HW_TW_Flow.html')

✔  Fig 2.1 saved → PIPG_2.1_HW_TW_Flow.html


In [10]:
# ════════════════════════════════════════════════════════════════════════
# PLOT 2.2 — PIPG: Breach Width
# ════════════════════════════════════════════════════════════════════════
fig22 = go.Figure()

fig22.add_trace(go.Scatter(
    x=[r['hr'] for r in pw], y=[r['width'] for r in pw],
    name='Breach Width (m)', mode='lines',
    line=dict(color=C_BW, width=2.8),
    fill='tozeroy', fillcolor='rgba(106,27,154,0.10)'
))

# Max breach width annotation
fig22.add_annotation(
    x=P_MAX_BW['hr'], y=P_MAX_BW['width'],
    text=f"<b>Max Width = {P_MAX_BW['width']:.1f} m<br>@ {hm_short(P_MAX_BW['hr'])}</b>",
    showarrow=True, arrowhead=2, arrowcolor=C_BW, ax=70, ay=-45,
    font=dict(size=10.5, color=C_BW),
    bgcolor='rgba(255,255,255,0.88)', bordercolor=C_BW, borderwidth=1)

# Phase labels
fig22.add_annotation(x=1.5, y=45,
    text='▶ Rapid Pipe<br>   Enlargement',
    showarrow=False, font=dict(size=9, color=C_BW),
    bgcolor='rgba(237,231,246,0.85)', bordercolor=C_BW, borderwidth=1)
fig22.add_annotation(x=20, y=70,
    text=f'Stable width<br>{P_MAX_BW["width"]:.0f} m',
    showarrow=False, font=dict(size=9, color='#555'),
    bgcolor='rgba(245,245,245,0.85)', bordercolor='#999', borderwidth=1)

# Piping init vline
fig22.add_vline(x=P_INIT['hr'], line_dash='dashdot',
                line_color='#F57F17', line_width=1.5,
                annotation_text=f"Piping init {hm_short(P_INIT['hr'])}",
                annotation_position='top right',
                annotation_font=dict(size=9, color='#F57F17'))

add_6hr_lines(fig22)
fig22.update_layout(**_base_layout(
    '<b>JAWALGAON DAM — PIPG FAILURE (2.2) : BREACH WIDTH DEVELOPMENT</b><br>'
    '<span style="font-size:11px">Internal Erosion Piping Tunnel Enlargement | '
    'REF: DBA-JWLG-PIPG-2.2</span>',
    w=1300, h=520
))
fig22.update_xaxes(
    title_text='Elapsed Time (hr)', dtick=6, range=[-0.5, 73],
    minor=dict(showgrid=True, gridcolor=GRID_MI, dtick=1),
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
fig22.update_yaxes(
    title_text='Breach Width (m)', dtick=10, range=[-1, 92],
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5),
    minor=dict(showgrid=True, gridcolor=GRID_MI, dtick=2))
title_block(fig22,
    f'DAM: Jawalgaon Dam | FAILURE: Piping (PIPG) | '
    f'Max Breach Width = {P_MAX_BW["width"]:.1f} m @ {hm_short(P_MAX_BW["hr"])} | '
    f'Full breach in {hm_short(P_MAX_BW["hr"])} from initiation',
    'SCALE: H:1cm=6hr  V:1cm=10m | REF: DBA-JWLG-PIPG-2.2')
fig22.write_html('PIPG_2.2_Breach_Width.html')
fig22.show()
print('✔  Fig 2.2 saved → PIPG_2.2_Breach_Width.html')

✔  Fig 2.2 saved → PIPG_2.2_Breach_Width.html


In [11]:
# ════════════════════════════════════════════════════════════════════════
# PLOT 2.3 — PIPG: Breach Velocity
# ════════════════════════════════════════════════════════════════════════
fig23 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[
        '(a) Breach Velocity (m/s) — Piping Channel',
        '(b) Breach Velocity vs Total Discharge (dual axis)'
    ],
    vertical_spacing=0.12, row_heights=[0.50, 0.50],
    specs=[[{'secondary_y': False}], [{'secondary_y': True}]]
)

# Velocity trace
fig23.add_trace(go.Scatter(
    x=[r['hr'] for r in pv], y=[r['velocity'] for r in pv],
    name='Breach Velocity (m/s)', mode='lines',
    line=dict(color=C_VEL, width=2.5),
    fill='tozeroy', fillcolor='rgba(0,105,92,0.10)'
), row=1, col=1)

# Max velocity annotation
fig23.add_annotation(
    x=P_MAX_VEL['hr'], y=P_MAX_VEL['velocity'],
    text=f"<b>Max Velocity = {P_MAX_VEL['velocity']:.3f} m/s<br>@ {hm_short(P_MAX_VEL['hr'])}</b>",
    showarrow=True, arrowhead=2, arrowcolor=C_VEL, ax=80, ay=-45,
    font=dict(size=10.5, color=C_VEL),
    bgcolor='rgba(255,255,255,0.88)', bordercolor=C_VEL, borderwidth=1,
    row=1, col=1)

# Critical velocity threshold (example: ~1.5 m/s = critical erosion velocity)
fig23.add_hline(y=1.5, line_dash='dot', line_color='#EF5350', line_width=1.5,
                annotation_text='Critical erosion threshold ≈ 1.5 m/s',
                annotation_position='top right',
                annotation_font=dict(size=9, color='#C62828'), row=1, col=1)

# Dual axis panel: velocity + discharge
fig23.add_trace(go.Scatter(
    x=[r['hr'] for r in pm], y=[r['total_q'] for r in pm],
    name='Total Discharge (m³/s) [Right axis]', mode='lines',
    line=dict(color=C_PIPG, width=2, dash='dot')
), row=2, col=1, secondary_y=True)

fig23.add_trace(go.Scatter(
    x=[r['hr'] for r in pv], y=[r['velocity'] for r in pv],
    name='Breach Velocity (m/s) [Left axis]', mode='lines',
    line=dict(color=C_VEL, width=2.2)
), row=2, col=1, secondary_y=False)

add_6hr_lines(fig23)
fig23.update_layout(**_base_layout(
    '<b>JAWALGAON DAM — PIPG FAILURE (2.3) : BREACH VELOCITY</b><br>'
    f'<span style="font-size:11px">Max Velocity = {P_MAX_VEL["velocity"]:.3f} m/s @ '
    f'{hm_short(P_MAX_VEL["hr"])} | REF: DBA-JWLG-PIPG-2.3</span>',
    w=1300, h=640
))
fig23.update_xaxes(
    title_text='Elapsed Time (hr)', dtick=6, range=[-0.5, 73],
    minor=dict(showgrid=True, gridcolor=GRID_MI, dtick=1),
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5), row=2, col=1)
fig23.update_yaxes(
    title_text='Velocity (m/s)', dtick=0.5, row=1, col=1,
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
fig23.update_yaxes(title_text='Velocity (m/s)', row=2, col=1, secondary_y=False, dtick=0.5)
fig23.update_yaxes(title_text='Discharge Q (m³/s)', row=2, col=1, secondary_y=True, dtick=200)
title_block(fig23,
    f'DAM: Jawalgaon Dam | FAILURE: Piping (PIPG) | '
    f'Max Velocity = {P_MAX_VEL["velocity"]:.3f} m/s @ {hm_short(P_MAX_VEL["hr"])} | '
    f'Erosion exceeds critical threshold during H+00 to H+06',
    'SCALE: H:1cm=6hr  V:1cm=0.5m/s | REF: DBA-JWLG-PIPG-2.3')
fig23.write_html('PIPG_2.3_Breach_Velocity.html')
fig23.show()
print('✔  Fig 2.3 saved → PIPG_2.3_Breach_Velocity.html')

✔  Fig 2.3 saved → PIPG_2.3_Breach_Velocity.html


In [12]:
# ════════════════════════════════════════════════════════════════════════
# PLOT 2.4 — PIPG: Downstream Hydrograph + Flood Arrival Time
# ════════════════════════════════════════════════════════════════════════
fig24 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[
        '(a) Downstream Flood Discharge (m³/s) — with Flood Arrival Markers',
        '(b) Downstream Stage / Water Surface Elevation (m)'
    ],
    vertical_spacing=0.10, row_heights=[0.60, 0.40]
)

# DS flow
fig24.add_trace(go.Scatter(
    x=[r['hr'] for r in pds], y=[r['ds_q'] for r in pds],
    name='Downstream Flood Q (m³/s)', mode='lines',
    line=dict(color=C_DS, width=2.8),
    fill='tozeroy', fillcolor='rgba(46,125,50,0.12)'
), row=1, col=1)

# Upstream reference
fig24.add_trace(go.Scatter(
    x=[r['hr'] for r in pm], y=[r['total_q'] for r in pm],
    name='Upstream Breach Q (m³/s) [reference]', mode='lines',
    line=dict(color=C_PIPG, width=1.8, dash='dot'), opacity=0.65
), row=1, col=1)

# Peak DS
pds_i = max(range(len(pds)), key=lambda i: pds[i]['ds_q'])
fig24.add_annotation(
    x=pds[pds_i]['hr'], y=pds[pds_i]['ds_q'],
    text=f"<b>DS Peak Q = {pds[pds_i]['ds_q']:.0f} m³/s<br>@ {hm_short(pds[pds_i]['hr'])}</b>",
    showarrow=True, arrowhead=2, arrowcolor=C_DS, ax=-70, ay=-48,
    font=dict(size=10.5, color=C_DS),
    bgcolor='rgba(255,255,255,0.88)', bordercolor=C_DS, borderwidth=1,
    row=1, col=1)

# ── FLOOD ARRIVAL ANNOTATIONS ──────────────────────────────────────────
# First DS arrival
fig24.add_vline(x=P_DS_INIT['hr'], line_dash='dashdot',
                line_color='#FF8F00', line_width=1.8)
fig24.add_annotation(
    x=P_DS_INIT['hr'], y=0.62, yref='paper',
    text=f"<b>▼ Flood Arrives DS</b><br>{hm_short(P_DS_INIT['hr'])}",
    showarrow=False, font=dict(size=9, color='#FF8F00'),
    bgcolor='rgba(255,248,225,0.90)', bordercolor='#FF8F00', borderwidth=1)

# Pipe initiation
fig24.add_vline(x=P_INIT['hr'], line_dash='dashdot',
                line_color='#F57F17', line_width=1.5)
fig24.add_annotation(
    x=P_INIT['hr'], y=0.90, yref='paper',
    text=f"<b>▲ Piping Starts</b><br>{hm_short(P_INIT['hr'])}",
    showarrow=False, font=dict(size=9, color='#F57F17'),
    bgcolor='rgba(255,253,231,0.90)', bordercolor='#F57F17', borderwidth=1)

# Travel time arrow
fig24.add_annotation(
    x=P_INIT['hr'], y=pds[pds_i]['ds_q'] * 0.35,
    ax=P_MAX_DS['hr'], ay=pds[pds_i]['ds_q'] * 0.35,
    xref='x', yref='y', axref='x', ayref='y',
    text=f"  Travel Time: {hm(PIPG_TRAVEL_HR)}",
    showarrow=True, arrowhead=2, arrowside='end+start',
    arrowcolor='#5C6BC0', arrowwidth=2,
    font=dict(size=10, color='#3949AB', family='Arial'),
    bgcolor='rgba(232,234,246,0.92)', bordercolor='#5C6BC0', borderwidth=1,
    row=1, col=1)

# Attenuation lag arrow
fig24.add_annotation(
    x=P_PEAK['hr'], y=pds[pds_i]['ds_q'] * 0.65,
    ax=P_MAX_DS['hr'], ay=pds[pds_i]['ds_q'] * 0.65,
    xref='x', yref='y', axref='x', ayref='y',
    text=f"  Peak Lag: {hm(PIPG_LAG_HR)}",
    showarrow=True, arrowhead=2, arrowside='end+start',
    arrowcolor='#00838F', arrowwidth=2,
    font=dict(size=10, color='#006064', family='Arial'),
    bgcolor='rgba(224,247,250,0.92)', bordercolor='#00838F', borderwidth=1,
    row=1, col=1)

# DS elevation
fig24.add_trace(go.Scatter(
    x=[r['hr'] for r in pds], y=[r['ds_elev'] for r in pds],
    name='DS Water Surface Elevation (m)', mode='lines',
    line=dict(color='#0277BD', width=2.2),
    fill='tozeroy', fillcolor='rgba(2,119,189,0.08)'
), row=2, col=1)

add_6hr_lines(fig24)
fig24.update_layout(**_base_layout(
    '<b>JAWALGAON DAM — PIPG FAILURE (2.4) : DOWNSTREAM FLOOD HYDROGRAPH</b><br>'
    f'<span style="font-size:11px">Flood Travel Time = {hm(PIPG_TRAVEL_HR)} | '
    f'Peak Attenuation Lag = {hm(PIPG_LAG_HR)} | REF: DBA-JWLG-PIPG-2.4</span>',
    w=1300, h=660
))
fig24.update_xaxes(
    title_text='Elapsed Time (hr)', dtick=6, range=[-0.5, 73],
    minor=dict(showgrid=True, gridcolor=GRID_MI, dtick=1),
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5), row=2, col=1)
fig24.update_yaxes(
    title_text='Discharge Q (m³/s)', dtick=100, row=1, col=1,
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
fig24.update_yaxes(
    title_text='Elevation (m)', row=2, col=1,
    showgrid=True, gridcolor=GRID, linecolor='#2a2a2a',
    linewidth=1.8, mirror=True, ticks='outside',
    tickfont=dict(family='Courier New', size=9.5))
title_block(fig24,
    f'DAM: Jawalgaon Dam | DS Peak Q = {P_MAX_DS["ds_q"]:.0f} m³/s @ {hm_short(P_MAX_DS["hr"])} | '
    f'Flood Travel Time = {hm(PIPG_TRAVEL_HR)} | Peak Lag = {hm(PIPG_LAG_HR)}',
    'SCALE: H:1cm=6hr  V:1cm=100m³/s | REF: DBA-JWLG-PIPG-2.4')
fig24.write_html('PIPG_2.4_Downstream_Hydrograph.html')
fig24.show()
print('✔  Fig 2.4 saved → PIPG_2.4_Downstream_Hydrograph.html')

✔  Fig 2.4 saved → PIPG_2.4_Downstream_Hydrograph.html


In [13]:
# ════════════════════════════════════════════════════════════════════════
# CELL — FLOOD ARRIVAL TIME SUMMARY TABLE  (formatted report)
# ════════════════════════════════════════════════════════════════════════
from IPython.display import display
import pandas as pd

df_arrival = pd.DataFrame({
    'Event / Parameter': [
        'Breach / Piping Initiation',
        'Upstream Peak Discharge',
        'Downstream Flood First Arrival',
        'Downstream Peak Discharge',
        '─── Flood Travel Time ───',
        'Breach Start → DS Peak (Travel Time)',
        'US Peak → DS Peak (Attenuation Lag)',
        '─── Magnitudes ──────────',
        'Upstream Peak Q (m³/s)',
        'Downstream Peak Q (m³/s)',
        'Attenuation Ratio (DS/US Peak)',
        'Max Breach Width (m)',
        'Max Breach Velocity (m/s)',
        'Max HW Elevation (m)',
    ],
    'OVTP (Overtopping)': [
        hm(O_INIT['hr']),
        hm(O_PEAK['hr']),
        hm(O_DS_INIT['hr']),
        hm(O_MAX_DS['hr']),
        '──────────────────',
        f"{hm(OVTP_TRAVEL_HR)}  ({OVTP_TRAVEL_HR:.2f} hr)",
        f"{hm(OVTP_LAG_HR)}  ({OVTP_LAG_HR:.2f} hr)",
        '──────────────────',
        f"{O_PEAK['total_q']:.2f}",
        f"{O_MAX_DS['ds_q']:.2f}",
        f"{O_MAX_DS['ds_q']/O_PEAK['total_q']:.3f}",
        f"{O_MAX_BW['width']:.1f}",
        f"{O_MAX_VEL['velocity']:.3f}",
        f"{O_MAX_HW['hw']:.3f}",
    ],
    'PIPG (Piping)': [
        hm(P_INIT['hr']),
        hm(P_PEAK['hr']),
        hm(P_DS_INIT['hr']),
        hm(P_MAX_DS['hr']),
        '──────────────────',
        f"{hm(PIPG_TRAVEL_HR)}  ({PIPG_TRAVEL_HR:.2f} hr)",
        f"{hm(PIPG_LAG_HR)}  ({PIPG_LAG_HR:.2f} hr)",
        '──────────────────',
        f"{P_PEAK['total_q']:.2f}",
        f"{P_MAX_DS['ds_q']:.2f}",
        f"{P_MAX_DS['ds_q']/P_PEAK['total_q']:.3f}",
        f"{P_MAX_BW['width']:.1f}",
        f"{P_MAX_VEL['velocity']:.3f}",
        f"{P_MAX_HW['hw']:.3f}",
    ]
})

styled = df_arrival.style\
    .set_caption('📋  Jawalgaon Dam — DBA Flood Arrival Time & Key Statistics Summary')\
    .set_table_styles([
        {'selector':'caption','props':[('font-size','14px'),('font-weight','bold'),
                                        ('color','#1A237E'),('padding','10px 0')]},
        {'selector':'th','props':[('background-color','#1565C0'),('color','white'),
                                   ('font-family','Arial'),('font-size','12px'),
                                   ('padding','8px 12px'),('text-align','center')]},
        {'selector':'td','props':[('font-family','Courier New'),('font-size','11.5px'),
                                   ('padding','6px 12px'),('border','1px solid #ccc')]},
    ])\
    .applymap(lambda v: 'background-color:#E3F2FD; font-weight:bold'
              if 'Travel' in str(v) or 'Lag' in str(v) else '',
              subset=['Event / Parameter'])\
    .set_properties(**{'text-align': 'left'})

display(styled)
print('\n✔  Flood Arrival Time Summary displayed')

/tmp/ipykernel_8178/2319505051.py:69: FutureWarning:

Styler.applymap has been deprecated. Use Styler.map instead.



,Event / Parameter,OVTP (Overtopping),PIPG (Piping)
0,Breach / Piping Initiation,14h 08m 25s,0h 01m 05s
1,Upstream Peak Discharge,19h 34m 40s,2h 50m 26s
2,Downstream Flood First Arrival,20h 08m 15s,8h 00m 56s
3,Downstream Peak Discharge,23h 15m 20s,9h 43m 22s
4,─── Flood Travel Time ───,──────────────────,──────────────────
5,Breach Start → DS Peak (Travel Time),9h 06m 55s (9.12 hr),9h 42m 16s (9.70 hr)
6,US Peak → DS Peak (Attenuation Lag),3h 40m 40s (3.68 hr),6h 52m 55s (6.88 hr)
7,─── Magnitudes ──────────,──────────────────,──────────────────
8,Upstream Peak Q (m³/s),4128.22,1567.10
9,Downstream Peak Q (m³/s),3808.01,925.41



✔  Flood Arrival Time Summary displayed


In [14]:
# ════════════════════════════════════════════════════════════════════════
# BONUS — Combined Overview: OVTP vs PIPG Downstream Comparison
# ════════════════════════════════════════════════════════════════════════
fig_cmp = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Upstream Breach Q — OVTP vs PIPG',
        'Downstream Flood Q — OVTP vs PIPG',
        'Reservoir HW Drawdown — OVTP vs PIPG',
        'Breach Width — OVTP vs PIPG'
    ],
    vertical_spacing=0.13, horizontal_spacing=0.10
)

# r1c1 — upstream Q
fig_cmp.add_trace(go.Scatter(x=[r['hr'] for r in om],
    y=[r['total_q'] for r in om], name='OVTP Upstream Q',
    line=dict(color=C_OVTP, width=2)), row=1, col=1)
fig_cmp.add_trace(go.Scatter(x=[r['hr'] for r in pm],
    y=[r['total_q'] for r in pm], name='PIPG Upstream Q',
    line=dict(color=C_PIPG, width=2, dash='dash')), row=1, col=1)

# r1c2 — downstream Q
fig_cmp.add_trace(go.Scatter(x=[r['hr'] for r in od],
    y=[r['ds_q'] for r in od], name='OVTP DS Q',
    line=dict(color=C_OVTP, width=2),
    fill='tozeroy', fillcolor='rgba(21,101,192,0.08)'), row=1, col=2)
fig_cmp.add_trace(go.Scatter(x=[r['hr'] for r in pds],
    y=[r['ds_q'] for r in pds], name='PIPG DS Q',
    line=dict(color=C_PIPG, width=2, dash='dash'),
    fill='tozeroy', fillcolor='rgba(183,28,28,0.08)'), row=1, col=2)

# r2c1 — HW drawdown
fig_cmp.add_trace(go.Scatter(x=[r['hr'] for r in om],
    y=[r['hw'] for r in om], name='OVTP HW',
    line=dict(color=C_OVTP, width=2), showlegend=False), row=2, col=1)
fig_cmp.add_trace(go.Scatter(x=[r['hr'] for r in pm],
    y=[r['hw'] for r in pm], name='PIPG HW',
    line=dict(color=C_PIPG, width=2, dash='dash'), showlegend=False), row=2, col=1)

# r2c2 — breach width
fig_cmp.add_trace(go.Scatter(x=[r['hr'] for r in ow],
    y=[r['width'] for r in ow], name='OVTP Breach Width',
    line=dict(color=C_OVTP, width=2), showlegend=False), row=2, col=2)
fig_cmp.add_trace(go.Scatter(x=[r['hr'] for r in pw],
    y=[r['width'] for r in pw], name='PIPG Breach Width',
    line=dict(color=C_PIPG, width=2, dash='dash'), showlegend=False), row=2, col=2)

# 6hr lines on all
for r in [1,2]:
    for c in [1,2]:
        for h in range(6,73,6):
            fig_cmp.add_vline(x=h, line_dash='dot',
                              line_color='rgba(90,90,90,0.25)', line_width=0.8)

fig_cmp.update_layout(
    title=dict(
        text='<b>JAWALGAON DAM — COMBINED DBA OVERVIEW: OVTP vs PIPG</b><br>'
             f'<span style="font-size:11px">'
             f'OVTP Travel Time: {hm(OVTP_TRAVEL_HR)}  |  '
             f'PIPG Travel Time: {hm(PIPG_TRAVEL_HR)}  |  '
             f'REF: DBA-JWLG-CMP-OVERVIEW</span>',
        font=dict(family='Arial', size=13), x=0.5, xanchor='center'),
    paper_bgcolor=PAPER, plot_bgcolor=BG,
    width=1400, height=780,
    margin=dict(l=70, r=50, t=100, b=100),
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#444',
                borderwidth=1.5, font=dict(family='Arial', size=11),
                x=0.99, y=0.99, xanchor='right', yanchor='top')
)
fig_cmp.update_xaxes(dtick=12, showgrid=True, gridcolor=GRID,
    linecolor='#2a2a2a', linewidth=1.5, mirror=True,
    ticks='outside', tickfont=dict(family='Courier New', size=8.5))
fig_cmp.update_yaxes(showgrid=True, gridcolor=GRID,
    linecolor='#2a2a2a', linewidth=1.5, mirror=True,
    ticks='outside', tickfont=dict(family='Courier New', size=8.5))
fig_cmp.update_xaxes(title_text='Elapsed Time (hr)', row=2, col=1)
fig_cmp.update_xaxes(title_text='Elapsed Time (hr)', row=2, col=2)
fig_cmp.update_yaxes(title_text='Q (m³/s)', row=1, col=1)
fig_cmp.update_yaxes(title_text='Q (m³/s)', row=1, col=2)
fig_cmp.update_yaxes(title_text='Elevation (m)', row=2, col=1)
fig_cmp.update_yaxes(title_text='Width (m)', row=2, col=2)
fig_cmp.write_html('Combined_OVTP_PIPG_Overview.html')
fig_cmp.show()
print('✔  Combined overview saved → Combined_OVTP_PIPG_Overview.html')
print()
print('═'*65)
print('  ALL 8 HYDROGRAPH PLOTS + COMBINED OVERVIEW COMPLETE')
print('═'*65)
print('  OVTP: 1.1_HW_TW_Flow  1.2_Breach_Width  1.3_Breach_Velocity  1.4_DS')
print('  PIPG: 2.1_HW_TW_Flow  2.2_Breach_Width  2.3_Breach_Velocity  2.4_DS')
print(f'  OVTP Flood Travel Time : {hm(OVTP_TRAVEL_HR)}')
print(f'  PIPG Flood Travel Time : {hm(PIPG_TRAVEL_HR)}')
print('═'*65)

✔  Combined overview saved → Combined_OVTP_PIPG_Overview.html

═════════════════════════════════════════════════════════════════
  ALL 8 HYDROGRAPH PLOTS + COMBINED OVERVIEW COMPLETE
═════════════════════════════════════════════════════════════════
  OVTP: 1.1_HW_TW_Flow  1.2_Breach_Width  1.3_Breach_Velocity  1.4_DS
  PIPG: 2.1_HW_TW_Flow  2.2_Breach_Width  2.3_Breach_Velocity  2.4_DS
  OVTP Flood Travel Time : 9h 06m 55s
  PIPG Flood Travel Time : 9h 42m 16s
═════════════════════════════════════════════════════════════════


In [17]:
import zipfile

output_files = [
    'OVTP_1.1_HW_TW_Flow.html',
    'OVTP_1.2_Breach_Width.html',
    'OVTP_1.3_Breach_Velocity.html',
    'OVTP_1.4_Downstream_Hydrograph.html',
    'PIPG_2.1_HW_TW_Flow.html',
    'PIPG_2.2_Breach_Width.html',
    'PIPG_2.3_Breach_Velocity.html',
    'PIPG_2.4_Downstream_Hydrograph.html',
    'Combined_OVTP_PIPG_Overview.html'
]

zip_filename = 'Jawalgaon_DBA_Graphs.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in output_files:
        try:
            zipf.write(file)
            print(f'Added {file} to {zip_filename}')
        except FileNotFoundError:
            print(f'Warning: {file} not found and was not added to the zip.')

print(f'\nSuccessfully created {zip_filename} containing all available graph files.')

Added OVTP_1.1_HW_TW_Flow.html to Jawalgaon_DBA_Graphs.zip
Added OVTP_1.2_Breach_Width.html to Jawalgaon_DBA_Graphs.zip
Added OVTP_1.3_Breach_Velocity.html to Jawalgaon_DBA_Graphs.zip
Added OVTP_1.4_Downstream_Hydrograph.html to Jawalgaon_DBA_Graphs.zip
Added PIPG_2.1_HW_TW_Flow.html to Jawalgaon_DBA_Graphs.zip
Added PIPG_2.2_Breach_Width.html to Jawalgaon_DBA_Graphs.zip
Added PIPG_2.3_Breach_Velocity.html to Jawalgaon_DBA_Graphs.zip
Added PIPG_2.4_Downstream_Hydrograph.html to Jawalgaon_DBA_Graphs.zip
Added Combined_OVTP_PIPG_Overview.html to Jawalgaon_DBA_Graphs.zip

Successfully created Jawalgaon_DBA_Graphs.zip containing all available graph files.


In [21]:
# ════════════════════════════════════════════════════════════════════════
# CELL — Install Playwright and browser dependencies for HTML to PNG conversion
# ════════════════════════════════════════════════════════════════════════
import subprocess, sys

print('Installing system dependencies for Playwright...')
subprocess.run(['apt-get', 'update'], check=False)
# Install common dependencies for headless Chrome on Linux
subprocess.run(['apt-get', 'install', '-y', 'libatk-bridge2.0-0', 'libcups2', 'libxcomposite1', 'libxdamage1', 'libxrandr2', 'libgtk-3-0', 'libgdk-pixbuf2.0-0', 'libfontconfig1', 'libatk1.0-0', 'libatk-bridge2.0-0', 'libxi6', 'libxkbcommon0', 'libxcursor1', 'libxshmfence1', 'libgbm1'], check=False)

print('Installing playwright...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'playwright', '--quiet'], check=False)
print('Installing browser drivers...')
# Install Chromium browser, as it's typically reliable for rendering
subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium'], check=False)
print('✔  Playwright and browser drivers ready')


Installing system dependencies for Playwright...
Installing playwright...
Installing browser drivers...
✔  Playwright and browser drivers ready


In [22]:
# ════════════════════════════════════════════════════════════════════════
# CELL — Function to convert HTML to PNG using Playwright
# ════════════════════════════════════════════════════════════════════════
from playwright.async_api import async_playwright
import os
import asyncio # Import asyncio

async def html_to_png(html_file_path, output_dir='.', max_width=1600, max_height=None):
    """Converts an HTML file to a PNG image using Playwright.

    Args:
        html_file_path (str): Path to the input HTML file.
        output_dir (str): Directory to save the PNG file. Defaults to current directory.
        max_width (int): Maximum width for the screenshot. Defaults to 1600 pixels.
        max_height (int): Maximum height for the screenshot. If None, it will capture full scrollable height.
    """
    png_file_name = os.path.basename(html_file_path).replace('.html', '.png')
    output_path = os.path.join(output_dir, png_file_name)

    print(f'Converting {html_file_path} to {output_path}...')

    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        await page.goto(f'file://{os.path.abspath(html_file_path)}', wait_until='networkidle')

        # Get the scrollable height of the page content
        if max_height is None:
            # Evaluate in browser context to get the full scroll height
            page_height = await page.evaluate('document.body.scrollHeight')
        else:
            page_height = max_height

        # Set viewport to capture full height or specified height
        await page.set_viewport_size({'width': max_width, 'height': page_height})

        await page.screenshot(path=output_path, full_page=True)
        await browser.close()
    print(f'✔  Successfully converted {html_file_path} to {png_file_name}')


# List of HTML files to convert (from previous execution)
html_output_files = [
    'OVTP_1.1_HW_TW_Flow.html',
    'OVTP_1.2_Breach_Width.html',
    'OVTP_1.3_Breach_Velocity.html',
    'OVTP_1.4_Downstream_Hydrograph.html',
    'PIPG_2.1_HW_TW_Flow.html',
    'PIPG_2.2_Breach_Width.html',
    'PIPG_2.3_Breach_Velocity.html',
    'PIPG_2.4_Downstream_Hydrograph.html',
    'Combined_OVTP_PIPG_Overview.html'
]

# Convert each HTML file to PNG
# Run the async function using asyncio.run or await it in an async cell
async def convert_all_html_to_png():
    for html_file in html_output_files:
        if os.path.exists(html_file):
            await html_to_png(html_file)
        else:
            print(f'Warning: HTML file not found: {html_file}. Skipping conversion.')

    print('\nConversion of all available HTML graphs to PNG complete.')

# Execute the async function
await convert_all_html_to_png()


Converting OVTP_1.1_HW_TW_Flow.html to ./OVTP_1.1_HW_TW_Flow.png...
✔  Successfully converted OVTP_1.1_HW_TW_Flow.html to OVTP_1.1_HW_TW_Flow.png
Converting OVTP_1.2_Breach_Width.html to ./OVTP_1.2_Breach_Width.png...
✔  Successfully converted OVTP_1.2_Breach_Width.html to OVTP_1.2_Breach_Width.png
Converting OVTP_1.3_Breach_Velocity.html to ./OVTP_1.3_Breach_Velocity.png...
✔  Successfully converted OVTP_1.3_Breach_Velocity.html to OVTP_1.3_Breach_Velocity.png
Converting OVTP_1.4_Downstream_Hydrograph.html to ./OVTP_1.4_Downstream_Hydrograph.png...
✔  Successfully converted OVTP_1.4_Downstream_Hydrograph.html to OVTP_1.4_Downstream_Hydrograph.png
Converting PIPG_2.1_HW_TW_Flow.html to ./PIPG_2.1_HW_TW_Flow.png...
✔  Successfully converted PIPG_2.1_HW_TW_Flow.html to PIPG_2.1_HW_TW_Flow.png
Converting PIPG_2.2_Breach_Width.html to ./PIPG_2.2_Breach_Width.png...
✔  Successfully converted PIPG_2.2_Breach_Width.html to PIPG_2.2_Breach_Width.png
Converting PIPG_2.3_Breach_Velocity.html to 

In [23]:
# ════════════════════════════════════════════════════════════════════════
# CELL — Create a zip file of all generated PNG images
# ════════════════════════════════════════════════════════════════════════
import zipfile
import os

png_output_files = [f.replace('.html', '.png') for f in html_output_files]
zip_png_filename = 'Jawalgaon_DBA_Graphs_PNG.zip'

with zipfile.ZipFile(zip_png_filename, 'w') as zipf:
    for file in png_output_files:
        if os.path.exists(file):
            zipf.write(file)
            print(f'Added {file} to {zip_png_filename}')
        else:
            print(f'Warning: {file} not found and was not added to the zip.')

print(f'\nSuccessfully created {zip_png_filename} containing all generated PNG files.')


Added OVTP_1.1_HW_TW_Flow.png to Jawalgaon_DBA_Graphs_PNG.zip
Added OVTP_1.2_Breach_Width.png to Jawalgaon_DBA_Graphs_PNG.zip
Added OVTP_1.3_Breach_Velocity.png to Jawalgaon_DBA_Graphs_PNG.zip
Added OVTP_1.4_Downstream_Hydrograph.png to Jawalgaon_DBA_Graphs_PNG.zip
Added PIPG_2.1_HW_TW_Flow.png to Jawalgaon_DBA_Graphs_PNG.zip
Added PIPG_2.2_Breach_Width.png to Jawalgaon_DBA_Graphs_PNG.zip
Added PIPG_2.3_Breach_Velocity.png to Jawalgaon_DBA_Graphs_PNG.zip
Added PIPG_2.4_Downstream_Hydrograph.png to Jawalgaon_DBA_Graphs_PNG.zip
Added Combined_OVTP_PIPG_Overview.png to Jawalgaon_DBA_Graphs_PNG.zip

Successfully created Jawalgaon_DBA_Graphs_PNG.zip containing all generated PNG files.
